# 图像编码 Demo（学生版）

请按教师提示先预测，再运行相应代码。

## 0. 工作目录检查

In [ ]:
from pathlib import Path

lesson_dir = Path.cwd()
assets_dir = lesson_dir / "assets"

assert assets_dir.is_dir(), (
    "当前 Jupyter working directory 不是本课目录："
    f"{lesson_dir}\n"
    "请从 1-2-encoding/1-2-3-image-encoding 目录启动 notebook。"
)

print("lesson directory:", lesson_dir)
print("assets directory:", assets_dir)


## 1. 生成固定测试图

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw

path = Path("assets/test-image.png")
path.parent.mkdir(parents=True, exist_ok=True)

w = h = 256
img = Image.new("RGB", (w, h))
px = img.load()

# 水平灰度渐变：用于观察 banding
for y in range(h):
    for x in range(w):
        v = round(255 * x / (w - 1))
        px[x, y] = (v, v, v)

draw = ImageDraw.Draw(img)

# 红、绿两个色块直接相接：接缝位于 x=72 / 73；蓝色作参照
draw.rectangle((12, 12, 72, 72), fill=(255, 0, 0))
draw.rectangle((73, 12, 133, 72), fill=(0, 255, 0))
draw.rectangle((184, 12, 244, 72), fill=(0, 0, 255))

# 下半部浅灰背景 + 大圆 + 粗斜线 + 细节条纹
draw.rectangle((0, 104, 128, 255), fill=(224, 224, 224))
draw.rectangle((136, 104, 255, 255), fill=(224, 224, 224))
draw.ellipse((0, 112, 128, 240), outline=(0, 0, 0), width=28)
draw.line((144, 240, 240, 144), fill=(0, 0, 0), width=28)
for x in [152, 160, 168, 176, 184, 192]:
    draw.line((x, 224, x, 244), fill=(255, 255, 255), width=2)

img.save(path)

assert img.size == (256, 256)
assert img.mode == "RGB"
assert path.is_file()

print("created:", path)


## 2. Demo A：Sampling

预测：从 256×256 到 8×8，哪些内容最先丢失？

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display

source_path = Path("assets/test-image.png")
assert source_path.is_file(), "请先运行固定测试图生成单元。"

src = Image.open(source_path).convert("RGB")
sizes = [256, 64, 32, 16, 8]

sampling_outputs = {}

for n in sizes:
    sampled = src.resize((n, n), resample=Image.Resampling.NEAREST)
    enlarged = sampled.resize((512, 512), resample=Image.Resampling.NEAREST)

    assert sampled.mode == "RGB"
    assert enlarged.mode == "RGB"
    assert enlarged.size == (512, 512)

    sampling_outputs[n] = enlarged

    print(f"{n} × {n}")
    display(enlarged)


## 3. Demo B：Quantization

预测：像素数量不变时，红绿接缝会一直看得见吗？

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display

source_path = Path("assets/test-image.png")
assert source_path.is_file()

src = Image.open(source_path).convert("RGB")
targets = [256, 16, 4, 2]
main_targets = [16, 2]
outputs = {}

for n in targets:
    q = src.quantize(
        colors=n,
        method=Image.Quantize.MEDIANCUT,
        dither=Image.Dither.NONE,
    ).convert("RGB")

    colors = q.getcolors(maxcolors=n + 1)
    assert colors is not None
    color_count = len(colors)

    assert q.size == src.size
    assert q.mode == "RGB"
    assert color_count <= n

    outputs[n] = q

print("original")
display(src)

for n in main_targets:
    print(f"max {n} colors")
    display(outputs[n])

# 256-color / 4-color 已保存在 outputs 中。需要追问时再 display(outputs[256]) / display(outputs[4])。


### 先解释，再验证

先用自己的话解释你看到的现象，再运行下一格。

In [ ]:
seam_left = (72, 40)
seam_right = (73, 40)

for n in [16, 2]:
    left_rgb = outputs[n].getpixel(seam_left)
    right_rgb = outputs[n].getpixel(seam_right)

    print(
        f"{n:2d} colors | "
        f"left={left_rgb} | right={right_rgb} | "
        f"same={left_rgb == right_rgb}"
    )


## 4. 一个 RGB 像素到 bits

In [ ]:
def byte_bits(x):
    return format(x, "08b")

rgb = (255, 128, 0)
print("RGB =", rgb)
print("bits:", " ".join(byte_bits(x) for x in rgb))


## 5. BMP 拓展（教师允许后再运行）

In [ ]:
from pathlib import Path
import struct

path = Path("assets/q7-3x2-24bit.bmp")
assert path.is_file(), f"找不到 {path}。请确认 Jupyter 从本课目录启动。"

data = path.read_bytes()

pixel_offset = struct.unpack_from("<I", data, 10)[0]
width = struct.unpack_from("<i", data, 18)[0]
height = struct.unpack_from("<i", data, 22)[0]
bits_per_pixel = struct.unpack_from("<H", data, 28)[0]
rows = abs(height)

color_value_bytes = width * rows * bits_per_pixel // 8
row_value_bytes = width * bits_per_pixel // 8
stored_row_bytes = ((width * bits_per_pixel + 31) // 32) * 4
pixel_array_bytes = stored_row_bytes * rows
file_bytes = len(data)

print("pixel-data offset :", pixel_offset, "bytes")
print("color values      :", color_value_bytes, "bytes")
print("raw row values    :", row_value_bytes, "bytes/row")
print("stored row size   :", stored_row_bytes, "bytes/row")
print("pixel array       :", pixel_array_bytes, "bytes")
print("whole file        :", file_bytes, "bytes")

assert pixel_offset == 54
assert color_value_bytes == 18
assert stored_row_bytes == 12
assert pixel_array_bytes == 24
assert file_bytes == 78
